# Parser di documenti per knowledge base

Progetto del corso *Programmazione con Python* — Master ProfessionAI.

Programma Python che, data una cartella di documenti in formati diversi (`.txt`, `.md`, `.csv`), ne estrae contenuto e metadati, li normalizza in una rappresentazione comune e costruisce un indice della knowledge base, salvato in JSON e interrogabile da riga di comando.

**Repository:** [https://github.com/massimiliano-cantore/kb-parser](https://github.com/massimiliano-cantore/kb-parser) — codice, moduli, archivio di prova e README con le istruzioni.

Il notebook è diviso in tre parti:

- **Parte A** — uso del programma da riga di comando sull'archivio di prova;
- **Parte B** — uso delle classi da Python: ricaricamento dell'indice, ricerche, estensione con un nuovo formato;
- **Parte C** — prova di robustezza su file problematici creati al momento, che non fanno parte dell'archivio di prova.

Per eseguirlo: *Runtime → Esegui tutte*.

## Installazione

Clonazione del repository e installazione del pacchetto. Il pacchetto usa solo la libreria standard, quindi non installa dipendenze esterne.

L'ultima riga rende il pacchetto importabile subito anche nelle celle Python: in Colab un'installazione con `pip install -e` diventa visibile al notebook solo dopo un riavvio del runtime.

In [1]:
import os
import sys

if not os.path.exists("/content/kb-parser"):
    !git clone -q https://github.com/massimiliano-cantore/kb-parser.git /content/kb-parser

%cd /content/kb-parser
!pip install -q -e .

if "/content/kb-parser/src" not in sys.path:
    sys.path.insert(0, "/content/kb-parser/src")

# Il codice sorgente

Il codice è organizzato come pacchetto Python installabile, in moduli con responsabilità separate. Questa sezione mostra la struttura del progetto e il contenuto di ogni modulo, così da poterlo leggere senza uscire dal notebook. Lo stesso codice è nel repository GitHub.

## Struttura del progetto

In [2]:
!find . -path ./.git -prune -o -path "./src/kb_parser.egg-info" -prune -o -name "__pycache__" -prune -o -type f -print | grep -v "^./sample_archive/" | sort
print()
!cat pyproject.toml
print()
!cat requirements.txt

./.gitattributes
./.gitignore
./CLAUDE.md
./README.md
./notebooks/consegna.ipynb
./notebooks/dev.ipynb
./pyproject.toml
./requirements-dev.txt
./requirements.txt
./src/kb_parser/__init__.py
./src/kb_parser/__main__.py
./src/kb_parser/cli.py
./src/kb_parser/document.py
./src/kb_parser/exceptions.py
./src/kb_parser/index.py
./src/kb_parser/readers/__init__.py
./src/kb_parser/readers/base.py
./src/kb_parser/readers/csv_reader.py
./src/kb_parser/readers/markdown_reader.py
./src/kb_parser/readers/txt_reader.py
./src/kb_parser/scanner.py
./tests/test_cli.py
./tests/test_document.py
./tests/test_index.py
./tests/test_readers.py
./tests/test_sample_archive.py
./tests/test_scanner.py



[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "kb-parser"
version = "0.1.0"
description = "Document parser and index builder for a knowledge base."
readme = "README.md"
requires-python = ">=3.10"
authors = [{ name = "Massimiliano Cantore" }]
dependencies = []

[project.scripts]
kb-parser = "kb_parser.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]



# Nessuna dipendenza esterna a runtime: il pacchetto usa solo la libreria standard.


## `src/kb_parser/document.py`

Modello del documento: la dataclass `Document`, la scheda comune a tutti i formati, con `to_dict()` e `from_dict()` per il JSON.

In [3]:
from IPython.display import Code

Code(filename="src/kb_parser/document.py", language="python")

"""Normalized document model shared by all readers.

Every reader, whatever the source format, turns a file into a ``Document``.
From that point on the rest of the program (index, search, statistics,
persistence) works only with ``Document`` objects and never needs to know
how the original file was written.
"""

from dataclasses import asdict, dataclass, fields
from typing import Any


@dataclass(frozen=True)
class Document:
    """A knowledge base document in normalized form.

    A ``Document`` is the common "index card" produced by every reader.
    It is immutable (``frozen=True``): once a file has been parsed, its card
    cannot be changed by mistake elsewhere in the program.

    Attributes:
        title: Document title. Taken from the content when possible,
            otherwise from the file name.
        date: First date found in the text, in ISO format ``YYYY-MM-DD``,
            or ``None`` if the text contains no date.
        word_count: Number of words in the normalized content.
        excerpt: Short excerpt of the content, used for previews.
        file_format: Source format, lowercase and without the dot
            (e.g. ``"txt"``, ``"md"``, ``"csv"``).
        path: Path of the source file, with forward slashes.
        content: Full normalized text of the document.
    """

    title: str
    date: str | None
    word_count: int
    excerpt: str
    file_format: str
    path: str
    content: str

    def to_dict(self) -> dict[str, Any]:
        """Convert the document into a JSON-serializable dictionary.

        Returns:
            A dictionary with one key per field. All values are strings,
            integers or ``None``, so it can be written with ``json.dump``.
        """
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "Document":
        """Rebuild a document from a dictionary loaded from JSON.

        Keys that do not correspond to a field are ignored, so an index
        written by a newer version of the program can still be loaded.

        Args:
            data: Dictionary produced by ``to_dict`` and read back from
                the index file.

        Returns:
            The rebuilt ``Document``.

        Raises:
            ValueError: If ``data`` is not a dictionary, a required field
                is missing, or a field has the wrong type.
        """
        if not isinstance(data, dict):
            raise ValueError(
                f"Invalid document entry: expected an object, got {type(data).__name__}"
            )

        field_names = [field.name for field in fields(cls)]
        missing = [name for name in field_names if name not in data]
        if missing:
            raise ValueError(
                f"Invalid document entry: missing field(s) {', '.join(missing)}"
            )

        values = {name: data[name] for name in field_names}
        _check_types(values)
        return cls(**values)


def _check_types(values: dict[str, Any]) -> None:
    """Verify that the values loaded from JSON have the expected types.

    Args:
        values: Field name to value mapping, one entry per ``Document`` field.

    Raises:
        ValueError: If a value has the wrong type.
    """
    text_fields = ("title", "excerpt", "file_format", "path", "content")
    for name in text_fields:
        if not isinstance(values[name], str):
            raise ValueError(_type_error(name, "a string", values[name]))

    if values["date"] is not None and not isinstance(values["date"], str):
        raise ValueError(_type_error("date", "a string or null", values["date"]))

    # bool is a subclass of int in Python, so it must be excluded explicitly.
    word_count = values["word_count"]
    if isinstance(word_count, bool) or not isinstance(word_count, int):
        raise ValueError(_type_error("word_count", "an integer", word_count))
    if word_count < 0:
        raise ValueError(f"Invalid document entry: word_count is negative ({word_count})")




## `src/kb_parser/exceptions.py`

Gerarchia delle eccezioni: una classe per ogni tipo di problema, tutte derivate da `KBParserError`.

In [4]:
from IPython.display import Code

Code(filename="src/kb_parser/exceptions.py", language="python")

"""Exception hierarchy for kb_parser.

Every problem the parser knows how to handle has its own exception class,
all derived from ``KBParserError``. The scanner catches ``KBParserError``
once, records the file and the reason in the scan report and moves on to
the next file.
"""

from pathlib import Path


class KBParserError(Exception):
    """Base class for all errors raised by kb_parser.

    Attributes:
        path: The file or folder the error refers to.
        message: Human readable description of the problem.
        reason: Short label used to group errors in the scan report.
    """

    reason = "parser error"

    def __init__(self, path: Path, message: str) -> None:
        """Initialize the error.

        Args:
            path: The file or folder the error refers to.
            message: Human readable description of the problem.
        """
        super().__init__(f"{path}: {message}")
        self.path = path
        self.message = message


class UnsupportedFormatError(KBParserError):
    """The file extension is not handled by any registered reader."""

    reason = "unsupported format"


class EmptyDocumentError(KBParserError):
    """The file is empty or contains no usable text."""

    reason = "empty file"


class DocumentReadError(KBParserError):
    """The file cannot be read (missing, no permission, too large, I/O error)."""

    reason = "unreadable file"


class CorruptedContentError(KBParserError):
    """The file can be read but its content is malformed or binary."""

    reason = "corrupted content"


class InvalidArchiveError(KBParserError):
    """The archive folder given by the user does not exist or is not a folder."""

    reason = "invalid archive"


class IndexFileError(KBParserError):
    """The index file cannot be written, read or is not a valid index."""

    reason = "invalid index"

## `src/kb_parser/readers/base.py`

`BaseReader`: classe astratta con i passi comuni a tutti i formati (lettura, codifica, controlli, titolo, data, estratto, conteggio parole). Le sottoclassi implementano solo `parse()`.

In [5]:
from IPython.display import Code

Code(filename="src/kb_parser/readers/base.py", language="python")

"""Abstract base class for all format readers.

``BaseReader.read`` implements the steps shared by every format (template
method pattern): reading the bytes, decoding, empty and binary checks,
title fallback, word count, excerpt and date extraction. Concrete readers
only implement ``parse``, the part that depends on the format.
"""

import re
import unicodedata
from abc import ABC, abstractmethod
from datetime import date
from pathlib import Path

from kb_parser.document import Document
from kb_parser.exceptions import (
    CorruptedContentError,
    DocumentReadError,
    EmptyDocumentError,
)

EXCERPT_LENGTH = 200
TITLE_MAX_LENGTH = 120
MAX_FILE_SIZE = 50 * 1024 * 1024  # 50 MB

# Share of control characters above which a decoded text is considered binary.
_MAX_CONTROL_RATIO = 0.05
# Control characters (Unicode category Cc) except tab, newlines, \v, \f and \r,
# which are normal in a text file.
_BINARY_CONTROLS = re.compile(r"[\x00-\x08\x0e-\x1f\x7f-\x9f]")
# Every control character except tab and newline, removed from the content.
_CONTENT_CONTROLS = re.compile(r"[\x00-\x08\x0b-\x1f\x7f-\x9f]")
# Invisible formatting characters (zero-width spaces, misplaced BOM).
_INVISIBLE = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")
_HORIZONTAL_SPACE = re.compile(r"[ \t\u00a0]{2,}|[\t\u00a0]")
_SPACE_AROUND_NEWLINE = re.compile(r" *\n *")
_EXTRA_BLANK_LINES = re.compile(r"\n{3,}")

_UTF16_BOMS = (b"\xff\xfe", b"\xfe\xff")

# One pattern per format, each starting with a digit and without look-behind:
# this lets the regex engine skip quickly over long texts. "Not preceded by
# a digit" is checked in _first_valid_date.
_ISO_DATE = re.compile(r"(\d{4})-(\d{2})-(\d{2})(?!\d)")
_EU_DATE = re.compile(r"(\d{1,2})/(\d{1,2})/(\d{4})(?!\d)")


class BaseReader(ABC):
    """Common interface and shared reading logic for all format readers.

    Subclasses declare the extensions they handle in ``extensions`` and
    implement ``parse``. Adding a new format means writing a new subclass
    and registering it, without touching the rest of the program.

    Attributes:
        extensions: Lowercase file extensions handled by the reader,
            including the leading dot (e.g. ``(".txt",)``).
    """

    extensions: tuple[str, ...] = ()

    def read(self, path: Path, root: Path | None = None) -> Document:
        """Read a file and turn it into a normalized ``Document``.

        Args:
            path: The file to read.
            root: Archive folder. When given, the document path is stored
                relative to it, so the index does not depend on the machine.

        Returns:
            The normalized document.

        Raises:
            DocumentReadError: If the file cannot be read or is too large.
            EmptyDocumentError: If the file contains no usable text.
            CorruptedContentError: If the content is binary or malformed.
        """
        raw = self._read_bytes(path)
        if not raw:
            raise EmptyDocumentError(path, "the file is empty")
        if not raw.strip():
            raise EmptyDocumentError(path, "the file contains only whitespace")

        text = self._decode(raw, path)
        if not text.strip():
            raise EmptyDocumentError(path, "the file contains only whitespace")

        title, content = self.parse(text, path)
        content = _normalize_whitespace(content)
        if not content:
            raise EmptyDocumentError(path, "the file contains no textual content")

        return Document(
            title=_clean_title(title) or path.stem,
            date=_find_date(content),
            word_count=len(content.split()),
            excerpt=_make_excerpt(content),
            file_format=path.suffix.lower().lstrip("."),
            path=_relative_path(path, root),
            content=content,
        )

    @abstractmethod
    def parse(self, text: str, path: Path) -> tuple[str | None, str]:
        """Extract title and content from the decoded text of a file.

        

## `src/kb_parser/readers/txt_reader.py`

Lettore dei file `.txt`.

In [6]:
from IPython.display import Code

Code(filename="src/kb_parser/readers/txt_reader.py", language="python")

"""Reader for plain text (.txt) files."""

from pathlib import Path

from kb_parser.readers.base import BaseReader


class TxtReader(BaseReader):
    """Reads plain text files.

    The title is the first non-empty line; the content is the whole text.
    """

    extensions = (".txt",)

    def parse(self, text: str, path: Path) -> tuple[str | None, str]:
        """Extract title and content from a plain text file.

        Args:
            text: The decoded file text.
            path: The file being read.

        Returns:
            The first non-empty line as title and the full text as content.
        """
        title = next((line.strip() for line in text.split("\n") if line.strip()), None)
        return title, text

## `src/kb_parser/readers/markdown_reader.py`

Lettore dei file `.md`: ricava il titolo e toglie la sintassi Markdown.

In [7]:
from IPython.display import Code

Code(filename="src/kb_parser/readers/markdown_reader.py", language="python")

"""Reader for Markdown (.md) files."""

import re
from pathlib import Path

from kb_parser.readers.base import BaseReader

_FRONT_MATTER = re.compile(r"\A---\n(.*?)\n(?:---|\.\.\.)\n", re.DOTALL)
_FRONT_MATTER_TITLE = re.compile(r"^title:\s*['\"]?(.+?)['\"]?\s*$", re.MULTILINE)
_HTML_COMMENT = re.compile(r"<!--.*?-->", re.DOTALL)
_TITLE_HEADING = re.compile(r"^#[ \t]+(.+?)[ \t]*#*[ \t]*$", re.MULTILINE)

# Line-level syntax, applied to each line.
_CODE_FENCE = re.compile(r"^\s*(```|~~~)")
_HEADING = re.compile(r"^\s{0,3}#{1,6}[ \t]*(.*?)[ \t]*#*[ \t]*$")
_BLOCKQUOTE = re.compile(r"^\s*(>\s?)+")
_LIST_MARKER = re.compile(r"^\s*(?:[-*+]|\d+[.)])\s+(?:\[[ xX]\]\s+)?")
_HORIZONTAL_RULE = re.compile(r"^\s*(?:[-*_=]\s*){3,}$")
_TABLE_SEPARATOR = re.compile(r"^\s*\|?\s*:?-+:?\s*(?:\|\s*:?-+:?\s*)*\|?\s*$")
_LINK_DEFINITION = re.compile(r"^\s*\[[^\]]+\]:\s*\S+")

# Inline syntax, applied to the whole text.
_IMAGE = re.compile(r"!\[([^\]]*)\]\([^)]*\)")
_LINK = re.compile(r"\[([^\]]+)\]\([^)]*\)")
_REFERENCE_LINK = re.compile(r"\[([^\]]+)\]\[[^\]]*\]")
_AUTOLINK = re.compile(r"<((?:https?|mailto):[^>]+)>")
_HTML_TAG = re.compile(r"</?[A-Za-z][^>]*>")
_INLINE_CODE = re.compile(r"`+([^`]+)`+")
_BOLD = re.compile(r"(\*\*|__)(?=\S)(.+?)(?<=\S)\1")
_STAR_ITALIC = re.compile(r"\*(?=\S)(.+?)(?<=\S)\*")
_UNDERSCORE_ITALIC = re.compile(r"(?<!\w)_(?=\S)(.+?)(?<=\S)_(?!\w)")
_STRIKETHROUGH = re.compile(r"~~(.+?)~~")


class MarkdownReader(BaseReader):
    """Reads Markdown files and strips the formatting syntax.

    The title is taken from the YAML front matter (``title:``) if present,
    otherwise from the first level-1 heading (``# Title``). The content is
    the text without Markdown symbols, so that searches and word counts are
    not affected by the formatting.
    """

    extensions = (".md", ".markdown")

    def parse(self, text: str, path: Path) -> tuple[str | None, str]:
        """Extract title and plain-text content from a Markdown file.

        Args:
            text: The decoded file text.
            path: The file being read.

        Returns:
            The title (or ``None``) and the content without Markdown syntax.
        """
        title = None
        front_matter = _FRONT_MATTER.match(text)
        if front_matter:
            title_match = _FRONT_MATTER_TITLE.search(front_matter.group(1))
            if title_match:
                title = title_match.group(1)
            text = text[front_matter.end():]

        text = _HTML_COMMENT.sub("", text)

        lines: list[str] = []
        in_code_block = False
        for line in text.split("\n"):
            if _CODE_FENCE.match(line):
                in_code_block = not in_code_block
                continue
            if in_code_block:
                # Code is kept as is: it is searchable, but a "# comment"
                # inside a code block must not become the title.
                lines.append(line)
                continue
            if title is None:
                heading = _TITLE_HEADING.match(line)
                if heading:
                    title = _strip_inline(heading.group(1))
            stripped = _strip_line(line)
            if stripped is not None:
                lines.append(stripped)

        return title, _strip_inline("\n".join(lines))


def _strip_line(line: str) -> str | None:
    """Remove line-level Markdown syntax.

    Args:
        line: A single line of the document, outside code blocks.

    Returns:
        The line without block markers, or ``None`` if the line carries
        no text (rules, table separators, link definitions).
    """
    if (
        _HORIZONTAL_RULE.match(line)
        or _TABLE_SEPARATOR.match(line)
        or _LINK_DEFINITION.match(line)
    ):
        return None
    heading = _HEADING.match(line)
    if heading:
        return heading.group(1)
    line = _BLOCKQUOTE.sub("", line)
    line = _LIST_MARKER.sub("", line)
    if "|" in line:
        line = " ".join(cell.strip() for cell in line.strip()

## `src/kb_parser/readers/csv_reader.py`

Lettore dei file `.csv`: riconosce il separatore, trasforma ogni riga in testo e scarta i file malformati.

In [8]:
from IPython.display import Code

Code(filename="src/kb_parser/readers/csv_reader.py", language="python")

"""Reader for CSV (.csv) files."""

import csv
import io
from pathlib import Path

from kb_parser.exceptions import CorruptedContentError, EmptyDocumentError
from kb_parser.readers.base import BaseReader

_CANDIDATE_DELIMITERS = ",;\t|"
_SNIFF_LINES = 20


class CsvReader(BaseReader):
    """Reads CSV files and turns each row into a readable line of text.

    The delimiter is detected automatically (Italian CSV files often use
    ``;``). The first non-empty row is the header; every data row becomes
    ``column: value; column: value``. A CSV has no natural title, so the
    file name is used.

    A row with a different number of fields than the header makes the
    whole file invalid: misaligned columns would produce wrong data.
    """

    extensions = (".csv",)

    def parse(self, text: str, path: Path) -> tuple[str | None, str]:
        """Extract the content of a CSV file as text.

        Args:
            text: The decoded file text.
            path: The file being read.

        Returns:
            ``None`` as title and one text line per data row as content.

        Raises:
            CorruptedContentError: If the CSV syntax is invalid or a row has
                the wrong number of fields.
            EmptyDocumentError: If the file has a header but no data rows.
        """
        dialect = _detect_dialect(text)
        reader = csv.reader(io.StringIO(text), dialect, strict=True)

        header: list[str] | None = None
        lines: list[str] = []
        try:
            for row in reader:
                if not any(cell.strip() for cell in row):
                    continue
                if header is None:
                    header = _clean_header(row)
                    continue
                if len(row) != len(header):
                    raise CorruptedContentError(
                        path,
                        f"line {reader.line_num} has {len(row)} fields, "
                        f"expected {len(header)} as in the header",
                    )
                lines.append(_row_to_text(header, row))
        except csv.Error as error:
            raise CorruptedContentError(
                path, f"invalid CSV syntax at line {reader.line_num} ({error})"
            ) from None

        if not lines:
            raise EmptyDocumentError(path, "the CSV file has no data rows")
        return None, "\n".join(lines)


def _detect_dialect(text: str) -> type[csv.Dialect] | csv.Dialect:
    """Detect the CSV dialect (mainly the delimiter) from the first lines.

    ``csv.Sniffer`` is tried first. It fails when rows are inconsistent,
    which is exactly the case of a malformed file; the delimiter is then
    taken from the header line, so that the wrong rows can be detected and
    reported instead of being read as a single column.

    Args:
        text: The decoded file text.

    Returns:
        The detected dialect. A file with no candidate delimiter in its
        header is read as a single comma-separated column.
    """
    sample = "\n".join(text.split("\n")[:_SNIFF_LINES])
    try:
        return csv.Sniffer().sniff(sample, delimiters=_CANDIDATE_DELIMITERS)
    except csv.Error:
        pass

    header_line = next((line for line in text.split("\n") if line.strip()), "")
    counts = {char: header_line.count(char) for char in _CANDIDATE_DELIMITERS}
    delimiter = max(counts, key=lambda char: counts[char])
    if counts[delimiter] == 0:
        return csv.excel

    class HeaderDialect(csv.excel):
        """Comma-separated dialect with the delimiter found in the header."""

    HeaderDialect.delimiter = delimiter
    return HeaderDialect


def _clean_header(row: list[str]) -> list[str]:
    """Clean header names, giving a name to empty columns.

    Args:
        row: The header row.

    Returns:
        The column names, stripped; empty names become ``column_N``.
    """
    return [cell.strip() or f"column_{number}" for number, cell in enumerate(row, start=1)]


def _row_to_te

## `src/kb_parser/readers/__init__.py`

Registro dei lettori: associa ogni estensione al suo lettore.

In [9]:
from IPython.display import Code

Code(filename="src/kb_parser/readers/__init__.py", language="python")

"""Format readers and the extension-to-reader registry.

The registry maps each file extension to the reader that handles it.
To support a new format, write a ``BaseReader`` subclass and pass it to
``register_reader``: nothing else in the program needs to change.
"""

from pathlib import Path

from kb_parser.exceptions import UnsupportedFormatError
from kb_parser.readers.base import BaseReader
from kb_parser.readers.csv_reader import CsvReader
from kb_parser.readers.markdown_reader import MarkdownReader
from kb_parser.readers.txt_reader import TxtReader

__all__ = [
    "BaseReader",
    "CsvReader",
    "MarkdownReader",
    "TxtReader",
    "get_reader",
    "register_reader",
    "supported_extensions",
]

_registry: dict[str, BaseReader] = {}


def register_reader(reader_class: type[BaseReader]) -> None:
    """Register a reader for all the extensions it declares.

    Args:
        reader_class: A ``BaseReader`` subclass.

    Raises:
        ValueError: If the reader declares no extension, or an extension is
            already handled by another reader.
    """
    if not reader_class.extensions:
        raise ValueError(f"{reader_class.__name__} declares no extensions")
    reader = reader_class()
    for extension in reader_class.extensions:
        extension = extension.lower()
        if extension in _registry:
            other = type(_registry[extension]).__name__
            raise ValueError(f"Extension '{extension}' is already handled by {other}")
        _registry[extension] = reader


def get_reader(path: Path) -> BaseReader:
    """Return the reader for a file, chosen by its extension.

    The extension is compared case-insensitively, so ``.TXT`` works too.

    Args:
        path: The file to read.

    Returns:
        The reader that handles the file extension.

    Raises:
        UnsupportedFormatError: If no reader handles the extension.
    """
    extension = path.suffix.lower()
    if not extension:
        raise UnsupportedFormatError(path, "the file has no extension")
    try:
        return _registry[extension]
    except KeyError:
        raise UnsupportedFormatError(
            path,
            f"extension '{extension}' is not supported "
            f"(supported: {', '.join(supported_extensions())})",
        ) from None


def supported_extensions() -> list[str]:
    """List the extensions handled by the registered readers.

    Returns:
        The extensions, sorted alphabetically.
    """
    return sorted(_registry)


for _reader_class in (TxtReader, MarkdownReader, CsvReader):
    register_reader(_reader_class)

## `src/kb_parser/scanner.py`

`ArchiveScanner` scorre la cartella e le sottocartelle; `ScanReport` raccoglie elaborati, scartati e motivi.

In [10]:
from IPython.display import Code

Code(filename="src/kb_parser/scanner.py", language="python")

"""Folder scanning and scan report.

``ArchiveScanner`` walks the archive folder and its subfolders, reads every
file with the right reader and collects the resulting documents. A file
that cannot be processed never stops the scan: it is recorded in the
``ScanReport`` with the reason, and the scan moves on to the next file.
"""

import os
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

from kb_parser.document import Document
from kb_parser.exceptions import DocumentReadError, InvalidArchiveError, KBParserError
from kb_parser.readers import get_reader

UNEXPECTED_ERROR_REASON = "unexpected error"


@dataclass(frozen=True)
class SkippedFile:
    """A file that could not be processed.

    Attributes:
        path: File path relative to the archive folder.
        reason: Short label used to group the errors (e.g. ``"empty file"``).
        message: Detailed description of the problem.
    """

    path: str
    reason: str
    message: str


@dataclass
class ScanReport:
    """Summary of a scan: processed files, skipped files and reasons.

    Attributes:
        root: The scanned archive folder.
        processed: Relative paths of the files turned into documents.
        skipped: The files that could not be processed.
    """

    root: Path
    processed: list[str] = field(default_factory=list)
    skipped: list[SkippedFile] = field(default_factory=list)

    @property
    def processed_count(self) -> int:
        """Number of files successfully turned into documents."""
        return len(self.processed)

    @property
    def skipped_count(self) -> int:
        """Number of files that could not be processed."""
        return len(self.skipped)

    @property
    def total_count(self) -> int:
        """Number of files found in the archive (hidden files excluded)."""
        return self.processed_count + self.skipped_count

    def skipped_by_reason(self) -> dict[str, int]:
        """Count the skipped files for each reason.

        Returns:
            Reason to count mapping, most frequent reasons first.
        """
        return dict(Counter(item.reason for item in self.skipped).most_common())

    def summary(self) -> str:
        """Build a human readable summary of the scan.

        Returns:
            A multi-line text with the counts and the list of skipped files
            with their reasons. The caller decides how to display it.
        """
        lines = [
            f"Archive: {self.root}",
            f"Files found: {self.total_count}",
            f"Processed:   {self.processed_count}",
            f"Skipped:     {self.skipped_count}",
        ]
        for reason, count in self.skipped_by_reason().items():
            lines.append(f"  - {reason}: {count}")
        if self.skipped:
            lines.append("")
            lines.append("Skipped files:")
            for item in self.skipped:
                lines.append(f"  - {item.path} [{item.reason}] {item.message}")
        return "\n".join(lines)


class ArchiveScanner:
    """Walks an archive folder and turns every supported file into a document.

    The scan is recursive and deterministic (folders and files are visited
    in alphabetical order). Hidden files and folders, whose name starts with
    a dot, are ignored. Symbolic links to folders are not followed, to avoid
    endless loops.
    """

    def __init__(self, root: Path | str) -> None:
        """Initialize the scanner.

        Args:
            root: The archive folder to scan.
        """
        self.root = Path(root)

    def scan(self) -> tuple[list[Document], ScanReport]:
        """Scan the archive and read all the files.

        Returns:
            The documents read successfully and the scan report.

        Raises:
            InvalidArchiveError: If the archive folder does not exist or is
                not a folder. This is the only error that stops the scan.
        """
        self._check_root()
        documents: list[

## `src/kb_parser/index.py`

`KnowledgeBaseIndex`: elenco, ricerca, filtro per formato, statistiche, salvataggio e caricamento del JSON.

In [11]:
from IPython.display import Code

Code(filename="src/kb_parser/index.py", language="python")

"""Knowledge base index: search, filters, statistics and JSON persistence.

``KnowledgeBaseIndex`` keeps the documents produced by the scanner and
answers the questions the knowledge base is built for: list the
documents, search them, filter them by format and summarize the archive.
It can be saved to a JSON file and loaded back without parsing the
original files again.
"""

import json
import re
import unicodedata
from collections import Counter
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Iterator

from kb_parser.document import Document
from kb_parser.exceptions import IndexFileError
from kb_parser.scanner import ScanReport, SkippedFile

INDEX_FORMAT_VERSION = 1
SORT_KEYS = ("path", "title", "date", "words")
DEFAULT_TOP = 5
SNIPPET_RADIUS = 60


@dataclass(frozen=True)
class SearchResult:
    """A document matching a search, with the data used to rank it.

    Attributes:
        document: The matching document.
        in_title: ``True`` if the keyword appears in the title.
        occurrences: Number of times the keyword appears in the content.
        snippet: Text around the first occurrence, with the match in
            square brackets.
    """

    document: Document
    in_title: bool
    occurrences: int
    snippet: str


@dataclass(frozen=True)
class IndexStatistics:
    """Summary figures of the knowledge base.

    Attributes:
        document_count: Number of indexed documents.
        total_words: Total number of words in all the documents.
        average_words: Average number of words per document.
        documents_by_format: Number of documents for each format.
        longest: The longest documents, longest first.
        with_date: Number of documents with a date.
        without_date: Number of documents without a date.
        oldest_date: Oldest document date, or ``None``.
        newest_date: Most recent document date, or ``None``.
        skipped_by_reason: Files skipped at build time, counted by reason.
    """

    document_count: int
    total_words: int
    average_words: float
    documents_by_format: dict[str, int]
    longest: tuple[Document, ...]
    with_date: int
    without_date: int
    oldest_date: str | None
    newest_date: str | None
    skipped_by_reason: dict[str, int]

    @property
    def skipped_count(self) -> int:
        """Number of files skipped at build time."""
        return sum(self.skipped_by_reason.values())


class KnowledgeBaseIndex:
    """The searchable collection of the documents of an archive.

    Attributes:
        source_folder: The archive folder the index was built from.
        created_at: Creation time in ISO format.
    """

    def __init__(
        self,
        documents: list[Document] | None = None,
        source_folder: str | None = None,
        created_at: str | None = None,
        skipped: list[SkippedFile] | None = None,
    ) -> None:
        """Initialize the index.

        Args:
            documents: The documents to index.
            source_folder: The archive folder the documents come from.
            created_at: Creation time in ISO format; defaults to now.
            skipped: Files that could not be processed at build time.

        Raises:
            ValueError: If two documents have the same path.
        """
        self._documents: list[Document] = []
        self._by_path: dict[str, Document] = {}
        for document in documents or []:
            self.add(document)
        self.source_folder = source_folder
        self.created_at = created_at or datetime.now().isoformat(timespec="seconds")
        self._skipped: list[SkippedFile] = list(skipped or [])

    @classmethod
    def from_scan(cls, documents: list[Document], report: ScanReport) -> "KnowledgeBaseIndex":
        """Build an index from the result of an archive scan.

        Args:
            documents: The documents returned by the scanner.
            report: The scan report, whose ski

## `src/kb_parser/cli.py`

Interfaccia da riga di comando con `argparse`: comandi `build`, `list`, `search`, `filter`, `stats`, `show`.

In [12]:
from IPython.display import Code

Code(filename="src/kb_parser/cli.py", language="python")

"""Command line interface for kb_parser.

This is the only module that prints: it turns the user's commands into
calls to the scanner and the index, and their results into text.

Examples:
    python -m kb_parser build sample_archive --output index.json
    python -m kb_parser search "manutenzione"
    python -m kb_parser stats --top 3
"""

import argparse
import os
import sys
from pathlib import Path

from kb_parser import __version__
from kb_parser.document import Document
from kb_parser.exceptions import KBParserError
from kb_parser.index import DEFAULT_TOP, SORT_KEYS, KnowledgeBaseIndex
from kb_parser.scanner import ArchiveScanner

DEFAULT_INDEX = "index.json"
TITLE_WIDTH = 45
PATH_WIDTH = 50

EXIT_OK = 0
EXIT_ERROR = 1


def main(argv: list[str] | None = None) -> int:
    """Run the command line interface.

    Args:
        argv: Command line arguments. Defaults to ``sys.argv[1:]``.

    Returns:
        The process exit code: 0 on success, 1 on error (2 for invalid
        arguments, set by ``argparse``).
    """
    _make_output_safe()
    args = build_parser().parse_args(argv)
    try:
        return args.handler(args)
    except (KBParserError, ValueError) as error:
        print(f"Error: {error}", file=sys.stderr)
        return EXIT_ERROR
    except BrokenPipeError:
        # The output was closed early (e.g. piped into "head"): not an error.
        # Redirect stdout so Python does not complain again while exiting.
        devnull = os.open(os.devnull, os.O_WRONLY)
        os.dup2(devnull, sys.stdout.fileno())
        return EXIT_OK


def build_parser() -> argparse.ArgumentParser:
    """Create the argument parser with all the subcommands.

    Returns:
        The configured parser.
    """
    parser = argparse.ArgumentParser(
        prog="kb_parser",
        description="Build and query a knowledge base index from a folder of documents.",
    )
    parser.add_argument("--version", action="version", version=f"%(prog)s {__version__}")

    index_option = argparse.ArgumentParser(add_help=False)
    index_option.add_argument(
        "--index", type=Path, default=Path(DEFAULT_INDEX),
        help=f"index file (default: {DEFAULT_INDEX})",
    )

    commands = parser.add_subparsers(dest="command", required=True, metavar="COMMAND")

    build = commands.add_parser("build", help="scan an archive folder and save the index")
    build.add_argument("archive", type=Path, help="folder containing the documents")
    build.add_argument(
        "--output", type=Path, default=Path(DEFAULT_INDEX),
        help=f"index file to write (default: {DEFAULT_INDEX})",
    )
    build.set_defaults(handler=_command_build)

    list_parser = commands.add_parser("list", parents=[index_option], help="list the documents")
    list_parser.add_argument(
        "--sort", choices=SORT_KEYS, default="path", help="sort order (default: path)"
    )
    list_parser.set_defaults(handler=_command_list)

    search = commands.add_parser(
        "search", parents=[index_option], help="search a keyword in titles and contents"
    )
    search.add_argument("keyword", help="text to look for (case-insensitive, also part of words)")
    search.set_defaults(handler=_command_search)

    filter_parser = commands.add_parser(
        "filter", parents=[index_option], help="list the documents of one format"
    )
    filter_parser.add_argument("format", help="file format, e.g. txt, md, csv")
    filter_parser.set_defaults(handler=_command_filter)

    stats = commands.add_parser("stats", parents=[index_option], help="show summary statistics")
    stats.add_argument(
        "--top", type=int, default=DEFAULT_TOP,
        help=f"number of longest documents to show (default: {DEFAULT_TOP})",
    )
    stats.set_defaults(handler=_command_stats)

    show = commands.add_parser("show", parents=[index_option], help="show a full document")
    show.add_argument("path", help="document path, as shown by the list command")
    show.set_defaults(handler=_command_show)



# Parte A — Riga di comando

## A.1 L'archivio di prova

Diciotto file fittizi nei tre formati, distribuiti in sottocartelle, con alcuni casi limite costruiti apposta: file vuoto, caratteri speciali, codifiche Latin-1 e Windows-1252, Markdown senza titolo, file molto lungo, estensione non prevista, CSV malformato, file binario con estensione `.txt` e un file nascosto. La descrizione di ogni caso è nel README.

In [13]:
!find sample_archive -type f | sort

sample_archive/.bozza_nascosta.txt
sample_archive/backup_corrotto.txt
sample_archive/file_vuoto.txt
sample_archive/manuale_piattaforma.pdf
sample_archive/note/appunti_riunione_fornitori.md
sample_archive/note/caratteri_speciali.txt
sample_archive/note/idee_nuovi_corsi.txt
sample_archive/procedure/accoglienza_nuovi_docenti.md
sample_archive/procedure/gestione_aule_laboratorio.md
sample_archive/procedure/procedura_backup_piattaforma.txt
sample_archive/registro_presenze_rotto.csv
sample_archive/schede_tecniche/catalogo_corsi_2026.csv
sample_archive/schede_tecniche/fornitori_windows.csv
sample_archive/schede_tecniche/inventario_laboratorio.csv
sample_archive/schede_tecniche/scheda_videoproiettore_aula3.md
sample_archive/storico/storico_interventi_manutenzione.txt
sample_archive/verbali/2025/verbale_comitato_didattico.md
sample_archive/verbali/2025/verbale_riunione_qualita_marzo.txt
sample_archive/verbali/verbale_collegio_docenti_2024.txt


## A.2 Costruzione dell'indice e riepilogo

Il comando `build` legge la cartella e le sottocartelle, applica a ogni file il lettore del suo formato e salva l'indice in `index.json`.

I file che non si possono elaborare vengono segnalati e saltati senza interrompere il processo. Al termine il programma stampa il riepilogo: documenti elaborati, documenti scartati e relative motivazioni.

In [14]:
!python -m kb_parser build sample_archive --output index.json

Archive: sample_archive
Files found: 18
Processed:   14
Skipped:     4
  - corrupted content: 2
  - empty file: 1
  - unsupported format: 1

Skipped files:
  - backup_corrotto.txt [corrupted content] the file contains null bytes, it looks like a binary file
  - file_vuoto.txt [empty file] the file is empty
  - manuale_piattaforma.pdf [unsupported format] extension '.pdf' is not supported (supported: .csv, .markdown, .md, .txt)
  - registro_presenze_rotto.csv [corrupted content] line 3 has 5 fields, expected 4 as in the header

Index saved to index.json (14 documents).


## A.3 L'indice salvato in JSON

Ogni documento è rappresentato con la stessa struttura, qualunque sia il formato di origine: titolo, data (se presente), numero di parole, estratto, formato, percorso e contenuto normalizzato.

In [15]:
!head -n 30 index.json

{
  "format_version": 1,
  "created_at": "2026-11-02T14:16:08",
  "source_folder": "sample_archive",
  "document_count": 14,
  "documents": [
    {
      "title": "appunti_riunione_fornitori",
      "date": null,
      "word_count": 42,
      "excerpt": "Appunti presi durante la telefonata con il fornitore dei banchi didattici. Disponibilità di due banchi aggiuntivi entro fine trimestre Il prezzo resta quello del listino in vigore Chiedere se la…",
      "file_format": "md",
      "path": "note/appunti_riunione_fornitori.md",
      "content": "Appunti presi durante la telefonata con il fornitore dei banchi didattici.\n\nDisponibilità di due banchi aggiuntivi entro fine trimestre\nIl prezzo resta quello del listino in vigore\nChiedere se la garanzia copre anche gli alimentatori\n\nDa ricontattare dopo la riunione di budget."
    },
    {
      "title": "Prova di caratteri speciali – àèéìòù ÀÈÉÌÒÙ ç ñ ß",
      "date": null,
      "word_count": 55,
      "excerpt": "Prova di caratteri sp

## A.4 Elenco dei documenti

Ordinamento predefinito per percorso; con `--sort` anche per titolo, data o lunghezza.

In [16]:
!python -m kb_parser list --index index.json

14 documents in index.json

PATH                                                FORMAT  DATE          WORDS  TITLE
note/appunti_riunione_fornitori.md                  md      -                42  appunti_riunione_fornitori
note/caratteri_speciali.txt                         txt     -                55  Prova di caratteri speciali – àèéìòù ÀÈÉÌÒÙ …
note/idee_nuovi_corsi.txt                           txt     -                44  Idee per nuovi corsi
procedure/accoglienza_nuovi_docenti.md              md      2025-09-01      122  Procedura di accoglienza dei nuovi docenti
procedure/gestione_aule_laboratorio.md              md      2026-01-15       90  Gestione delle aule e dei laboratori
procedure/procedura_backup_piattaforma.txt          txt     2025-11-20       98  Procedura di backup della piattaforma e-lear…
schede_tecniche/catalogo_corsi_2026.csv             csv     -                93  catalogo_corsi_2026
schede_tecniche/fornitori_windows.csv               csv     -                6

In [17]:
!python -m kb_parser list --sort date --index index.json

14 documents in index.json

PATH                                                FORMAT  DATE          WORDS  TITLE
storico/storico_interventi_manutenzione.txt         txt     2021-03-13    55497  Storico interventi di manutenzione 2019-2025
schede_tecniche/scheda_videoproiettore_aula3.md     md      2023-02-10       68  Scheda tecnica: videoproiettore aula 3
verbali/verbale_collegio_docenti_2024.txt           txt     2024-12-20       82  Verbale del collegio docenti
verbali/2025/verbale_riunione_qualita_marzo.txt     txt     2025-03-12       93  VERBALE RIUNIONE DEL GRUPPO QUALITÀ
verbali/2025/verbale_comitato_didattico.md          md      2025-06-18       60  Verbale del comitato didattico
procedure/accoglienza_nuovi_docenti.md              md      2025-09-01      122  Procedura di accoglienza dei nuovi docenti
procedure/procedura_backup_piattaforma.txt          txt     2025-11-20       98  Procedura di backup della piattaforma e-lear…
schede_tecniche/inventario_laboratorio.csv       

## A.5 Ricerca per parola chiave

La ricerca avviene nel titolo e nel contenuto, senza distinzione tra maiuscole e minuscole, e trova anche parti di parola. I risultati sono ordinati per rilevanza (prima i documenti con la parola nel titolo, poi per numero di occorrenze) e mostrano un frammento di testo con la parola tra parentesi quadre.

In [18]:
!python -m kb_parser search "manutenzione" --index index.json

2 documents contain 'manutenzione':

1. Storico interventi di manutenzione 2019-2025
   storico/storico_interventi_manutenzione.txt  (in title, 1 in content)
   Storico interventi di [manutenzione] 2019-2025 Registro completo degli interventi tecnici su at…

2. Scheda tecnica: videoproiettore aula 3
   schede_tecniche/scheda_videoproiettore_aula3.md  (1 in content)
   …ressi: HDMI ×2, VGA, LAN Data di installazione: 2023-02-10 [Manutenzione] Pulire il filtro dell'aria ogni 500 ore di utilizzo. La la…


In [19]:
!python -m kb_parser search "LABORATORIO" --index index.json

7 documents contain 'LABORATORIO':

1. inventario_laboratorio
   schede_tecniche/inventario_laboratorio.csv  (in title)
   codice: LAB-001; descrizione: Banco didattico PLC; quantità: 6; stato: operativo; ultima verifica: 10/01/2026 codice: LAB-002; descrizione: Quadro elettrico didattico; quantità: 2; stato: operativo…

2. Gestione delle aule e dei laboratori
   procedure/gestione_aule_laboratorio.md  (2 in content)
   …otazione Le aule si prenotano dal calendario condiviso. Il [laboratorio] di automazione (aula 3) richiede la presenza di un tecnico…

3. Idee per nuovi corsi
   note/idee_nuovi_corsi.txt  (1 in content)
   …orsi - Corso breve su Git e GitHub per i docenti tecnici - [Laboratorio] di robotica educativa per le scuole medie - Percorso serale…

4. Procedura di accoglienza dei nuovi docenti
   procedure/accoglienza_nuovi_docenti.md  (1 in content)
   …a. Il primo giorno il tutor d'aula accompagna il docente in [laboratorio]. Documenti da consegnare Curriculum aggiornato in f

## A.6 Filtro per formato

In [20]:
!python -m kb_parser filter csv --index index.json

3 documents with format 'csv'

PATH                                                FORMAT  DATE          WORDS  TITLE
schede_tecniche/catalogo_corsi_2026.csv             csv     -                93  catalogo_corsi_2026
schede_tecniche/fornitori_windows.csv               csv     -                61  fornitori_windows
schede_tecniche/inventario_laboratorio.csv          csv     2026-01-10       80  inventario_laboratorio


In [21]:
!python -m kb_parser filter md --index index.json

5 documents with format 'md'

PATH                                                FORMAT  DATE          WORDS  TITLE
note/appunti_riunione_fornitori.md                  md      -                42  appunti_riunione_fornitori
procedure/accoglienza_nuovi_docenti.md              md      2025-09-01      122  Procedura di accoglienza dei nuovi docenti
procedure/gestione_aule_laboratorio.md              md      2026-01-15       90  Gestione delle aule e dei laboratori
schede_tecniche/scheda_videoproiettore_aula3.md     md      2023-02-10       68  Scheda tecnica: videoproiettore aula 3
verbali/2025/verbale_comitato_didattico.md          md      2025-06-18       60  Verbale del comitato didattico


## A.7 Statistiche di sintesi

Documenti per tipo, documenti più lunghi, date e file scartati durante la costruzione.

In [22]:
!python -m kb_parser stats --index index.json

Index: index.json
Built from: sample_archive on 2026-11-02T14:16:08

Documents:        14
Total words:      56485
Average words:    4035
With a date:      9
Without a date:   5
Date range:       2021-03-13 - 2026-01-15

Documents by format:
  txt        6
  md         5
  csv        3

Longest documents (top 5):
     55497 words  storico/storico_interventi_manutenzione.txt
       122 words  procedure/accoglienza_nuovi_docenti.md
        98 words  procedure/procedura_backup_piattaforma.txt
        93 words  schede_tecniche/catalogo_corsi_2026.csv
        93 words  verbali/2025/verbale_riunione_qualita_marzo.txt

Files skipped at build time: 4
  - corrupted content: 2
  - empty file: 1
  - unsupported format: 1


## A.8 Scheda completa di un documento

Il verbale qui sotto è salvato in codifica Latin-1: le lettere accentate vengono lette correttamente.

In [23]:
!python -m kb_parser show verbali/verbale_collegio_docenti_2024.txt --index index.json

Title:   Verbale del collegio docenti
Path:    verbali/verbale_collegio_docenti_2024.txt
Format:  txt
Date:    2024-12-20
Words:   82
Excerpt: Verbale del collegio docenti Data: 20/12/2024 Il collegio si riunisce per l'analisi dell'anno formativo. Partecipano 14 docenti. Si discute della qualità dei materiali didattici: è emersa l'esigenza…

Verbale del collegio docenti

Data: 20/12/2024

Il collegio si riunisce per l'analisi dell'anno formativo. Partecipano 14 docenti.
Si discute della qualità dei materiali didattici: è emersa l'esigenza di uniformare le
dispense, perché oggi ciascun docente usa un modello proprio.
Il coordinatore propone un modello unico; la proposta è approvata all'unanimità.
Si ricorda che l'attività di laboratorio richiede la presenza del tecnico già alle 8.30.
Nota sul file: questo verbale è salvato in codifica Latin-1 (ISO-8859-1), come i documenti
prodotti dal vecchio gestionale.


## A.9 Ricaricamento dell'indice senza rifare il parsing

I comandi `list`, `search`, `filter`, `stats` e `show` leggono soltanto `index.json`. Per dimostrarlo, la cartella dei documenti viene rinominata temporaneamente: la ricerca funziona ugualmente, perché il programma non rilegge i file originali.

In [24]:
!mv sample_archive sample_archive_hidden
!python -m kb_parser search "verbale" --index index.json
!mv sample_archive_hidden sample_archive

3 documents contain 'verbale':

1. Verbale del collegio docenti
   verbali/verbale_collegio_docenti_2024.txt  (in title, 2 in content)
   [Verbale] del collegio docenti Data: 20/12/2024 Il collegio si riun…

2. Verbale del comitato didattico
   verbali/2025/verbale_comitato_didattico.md  (in title, 1 in content)
   [Verbale] del comitato didattico Riunione del 2025-06-18, modalità m…

3. VERBALE RIUNIONE DEL GRUPPO QUALITÀ
   verbali/2025/verbale_riunione_qualita_marzo.txt  (in title, 1 in content)
   [VERBALE] RIUNIONE DEL GRUPPO QUALITÀ Data: 12/03/2025 Luogo: sede d…


## A.10 Errori dell'utente

Un indice inesistente o una cartella sbagliata producono un messaggio chiaro, senza traceback, e un codice di uscita diverso da zero.

In [25]:
!python -m kb_parser stats --index indice_inesistente.json
!python -m kb_parser build cartella_inesistente

Error: indice_inesistente.json: the index file does not exist, run the 'build' command first


Error: cartella_inesistente: the archive folder does not exist


# Parte B — Uso da Python

## B.1 Ricaricamento dell'indice

`KnowledgeBaseIndex.load()` ricostruisce l'indice dal file JSON: ogni scheda torna a essere un oggetto `Document`, senza leggere di nuovo i documenti originali.

In [26]:
from kb_parser.index import KnowledgeBaseIndex

index = KnowledgeBaseIndex.load("index.json")
print(f"Documenti caricati da index.json: {len(index)}")
print(f"Indice creato il {index.created_at} dalla cartella '{index.source_folder}'")

document = index.get("procedure/accoglienza_nuovi_docenti.md")
print()
print(type(document).__name__)
print("title:      ", document.title)
print("date:       ", document.date)
print("word_count: ", document.word_count)
print("file_format:", document.file_format)
print("path:       ", document.path)
print("excerpt:    ", document.excerpt)

Documenti caricati da index.json: 14
Indice creato il 2026-11-02T14:16:08 dalla cartella 'sample_archive'

Document
title:       Procedura di accoglienza dei nuovi docenti
date:        2025-09-01
word_count:  122
file_format: md
path:        procedure/accoglienza_nuovi_docenti.md
excerpt:     Procedura di accoglienza dei nuovi docenti Codice: PRO-DID-04 · Revisione: 3 · Data di emissione: 2025-09-01 Scopo Definire i passaggi con cui Helix Formazione accoglie i docenti al primo incarico…


## B.2 Ricerca, filtro e statistiche

In [27]:
for result in index.search("filtro"):
    print(f"{result.document.title}  (nel titolo: {result.in_title}, occorrenze: {result.occurrences})")
    print(f"    {result.snippet}")

print()
print("Documenti CSV:", [doc.path for doc in index.filter_by_format("csv")])

stats = index.statistics(top=3)
print()
print("Documenti per formato:", stats.documents_by_format)
print("I tre più lunghi:     ", [(doc.path, doc.word_count) for doc in stats.longest])
print("Scartati alla build:  ", stats.skipped_by_reason)

Scheda tecnica: videoproiettore aula 3  (nel titolo: False, occorrenze: 1)
    …Data di installazione: 2023-02-10 Manutenzione Pulire il [filtro] dell'aria ogni 500 ore di utilizzo. La lampada va sostituit…

Documenti CSV: ['schede_tecniche/catalogo_corsi_2026.csv', 'schede_tecniche/fornitori_windows.csv', 'schede_tecniche/inventario_laboratorio.csv']

Documenti per formato: {'txt': 6, 'md': 5, 'csv': 3}
I tre più lunghi:      [('storico/storico_interventi_manutenzione.txt', 55497), ('procedure/accoglienza_nuovi_docenti.md', 122), ('procedure/procedura_backup_piattaforma.txt', 98)]
Scartati alla build:   {'corrupted content': 2, 'empty file': 1, 'unsupported format': 1}


## B.3 Estendere il programma con un nuovo formato

Ogni formato è una classe derivata da `BaseReader`, che contiene i passi comuni (lettura, codifica, controlli, data, estratto, conteggio delle parole). Una nuova classe deve solo dichiarare le estensioni e implementare `parse()`.

Qui si aggiunge il supporto ai file `.log` con una sola classe, senza modificare scanner, indice o interfaccia.

In [28]:
from pathlib import Path

from kb_parser.readers import BaseReader, register_reader, supported_extensions
from kb_parser.scanner import ArchiveScanner


class LogReader(BaseReader):
    """Reads .log files: the first line is the title, the content is the full text."""

    extensions = (".log",)

    def parse(self, text: str, path: Path) -> tuple[str | None, str]:
        first_line = text.strip().split("\n")[0]
        return first_line, text


demo = Path("/content/demo_log")
demo.mkdir(exist_ok=True)
(demo / "server.log").write_text(
    "Log del server della piattaforma\n2026-01-10 08:00 avvio\n2026-01-10 08:05 backup completato\n",
    encoding="utf-8",
)

_, report = ArchiveScanner(demo).scan()
print("Prima della registrazione:")
print(report.summary())

if ".log" not in supported_extensions():
    register_reader(LogReader)

documents, report = ArchiveScanner(demo).scan()
print()
print("Dopo la registrazione:")
print(report.summary())
print(documents[0].title, "|", documents[0].date, "|", documents[0].word_count, "parole")

Prima della registrazione:
Archive: /content/demo_log
Files found: 1
Processed:   0
Skipped:     1
  - unsupported format: 1

Skipped files:
  - server.log [unsupported format] extension '.log' is not supported (supported: .csv, .markdown, .md, .txt)

Dopo la registrazione:
Archive: /content/demo_log
Files found: 1
Processed:   1
Skipped:     0
Log del server della piattaforma | 2026-01-10 | 12 parole


# Parte C — Prova di robustezza

Il parser deve reggere su archivi che non sono stati preparati per lui. La cella seguente crea al momento una cartella con file problematici diversi da quelli dell'archivio di prova, insieme ad alcuni file validi, e la elabora: ogni problema viene segnalato con il suo motivo e l'elaborazione prosegue sugli altri file.

In [29]:
import random
from pathlib import Path

test = Path("/content/robustness_test")
(test / "sottocartella" / "più livelli").mkdir(parents=True, exist_ok=True)
rng = random.Random(0)

files = {
    # validi, ma insoliti
    "NOTE IN MAIUSCOLO.TXT": "Nota con estensione maiuscola e spazi nel nome.".encode("utf-8"),
    "unicode_windows.txt": "Nota salvata dal Blocco note di Windows in UTF-16".encode("utf-16"),
    "sottocartella/più livelli/annotazione è.md": "# Titolo con accenti\nCartelle annidate e nomi accentati.".encode("utf-8"),
    "solo_una_colonna.csv": "matricola\nM-001\nM-002\n".encode("utf-8"),
    # problematici
    "senza_estensione": b"Un file senza estensione",
    "documento.docx": b"PK\x03\x04 finto documento Word",
    "immagine_rinominata.txt": bytes(rng.randrange(256) for _ in range(2048)),
    "solo_spazi.md": b"   \n\t\n   ",
    "solo_simboli.md": b"---\n***\n___\n",
    "virgolette_aperte.csv": 'codice,nota\nA1,"nota senza chiusura\n'.encode("utf-8"),
    "solo_intestazione.csv": b"codice;descrizione\n",
    "colonne_sbagliate.csv": b"a;b;c\n1;2;3\n4;5\n",
}
for name, data in files.items():
    (test / name).write_bytes(data)

!python -m kb_parser build /content/robustness_test --output /content/robustness_index.json

Archive: /content/robustness_test
Files found: 12
Processed:   4
Skipped:     8
  - corrupted content: 3
  - empty file: 3
  - unsupported format: 2

Skipped files:
  - colonne_sbagliate.csv [corrupted content] line 3 has 2 fields, expected 3 as in the header
  - documento.docx [unsupported format] extension '.docx' is not supported (supported: .csv, .markdown, .md, .txt)
  - immagine_rinominata.txt [corrupted content] the file contains null bytes, it looks like a binary file
  - senza_estensione [unsupported format] the file has no extension
  - solo_intestazione.csv [empty file] the CSV file has no data rows
  - solo_simboli.md [empty file] the file contains no textual content
  - solo_spazi.md [empty file] the file contains only whitespace
  - virgolette_aperte.csv [corrupted content] invalid CSV syntax at line 2 (unexpected end of data)

Index saved to /content/robustness_index.json (4 documents).


In [30]:
!python -m kb_parser list --index /content/robustness_index.json

4 documents in /content/robustness_index.json

PATH                                                FORMAT  DATE          WORDS  TITLE
NOTE IN MAIUSCOLO.TXT                               txt     -                 8  Nota con estensione maiuscola e spazi nel no…
solo_una_colonna.csv                                csv     -                 4  solo_una_colonna
sottocartella/più livelli/annotazione è.md          md      -                 8  Titolo con accenti
unicode_windows.txt                                 txt     -                 9  Nota salvata dal Blocco note di Windows in U…
